# 13.8. API PyTorch более высокого уровня: краткое введение в PyTorch-Lightning 

В последние годы сообщество PyTorch разработало несколько различных библиотек и АРl-интерфейсов, работающих поверх PyTorch. К наиболее известным примерам относятся fastai8, Catalyst9, PyTorch Lightning10 и PyTorch-lgnite.

В этом разделе мы рассмотрим PyTorch Lightning (сокращенно Lightning) - широко используемую библиотеку PyTorch, которая упрощает обучение глубоких нейронных сетей за счет удаления большей части стандартного кода. Однако, несмотря на то, что
главное преимущество Lightning заключается в ее простоте и гибкости, она позволяет нам использовать множество дополнительных функций - таких как поддержка нескольких графических процессоров и быстрое обучение с низкой точностью, о чем вы
можете узнать в официальной документации по адресу: https://pytorch-lightning.rtfd.io/en/latest/.

Существует также введение в PyTorch-Ignite, расположенное по адресу: https://github.com/rasbt/machine-learning-book/blob/main/ch 13/ch13_part4_ignite.ipynb. Ранее мы реализовали многослойный персептрон для классификации рукописных цифр
в наборе данных МNIST. В следующих разделах мы повторно реализуем этот классификатор с помощью Lightning. 

### 13.8.1. Настройка модели PyTorch Lightning 

Начнем мы с построения модели, которую будем обучать в следующих разделах. Определить модель для Lightning относительно просто, поскольку она основана на обычном коде Python и PyTorch. Все, что требуется для реализации модели Lightning, - это использовать LightningMoctule вместо обычного модуля PyTorch. Чтобы воспользоваться удобными функциями PyTorch, такими как АРl-интерфейс обучателя и автоматическое ведение журнала, мы просто определим несколько методов со специальными именами:

In [1]:
import pytorch_lightning as pl
import torch 
import torch.nn as nn 

from torchmetrics import __version__ as torchmetrics_version
from pkg_resources import parse_version

from torchmetrics import Accuracy
# Из библиотеки torchmetrics импортирует класс Accuracy — готовую реализацию метрики точности для классификации.

W0716 20:29:20.639000 11084 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
C:\Users\Bushi\AppData\Local\Temp\ipykernel_11084\3721461691.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


In [2]:
class MultiLayerPerceptron(pl.LightningModule):
    def __init__(self, image_shape=(1, 28, 28), hidden_units=(32, 16)):
        super().__init__()
        
        if parse_version(torchmetrics_version) > parse_version("0.8"):
            self.train_acc = Accuracy(task="multiclass", num_classes=10)
            self.valid_acc = Accuracy(task="multiclass", num_classes=10)
            self.test_acc = Accuracy(task="multiclass", num_classes=10)
        else:
            self.train_acc = Accuracy()
            self.valid_acc = Accuracy()
            self.test_acc = Accuracy()
    
        input_size = image_shape[0] * image_shape[1] * image_shape[2] 
        all_layers = [nn.Flatten()]
        for hidden_unit in hidden_units: 
            layer = nn.Linear(input_size, hidden_unit) 
            all_layers.append(layer) 
            all_layers.append(nn.ReLU()) 
            input_size = hidden_unit 
 
        all_layers.append(nn.Linear(hidden_units[-1], 10)) 
        self.model = nn.Sequential(*all_layers)

# forward определяет, как данные проходят через модель.
# Просто передает вход x через self.model и возвращает результат (логиты).
# В Lightning этот метод вызывается, когда мы пишем self(x).
    def forward(self, x):
        x = self.model(x)
        return x

# training_step — это один шаг обучения на одном батче. Lightning вызывает этот метод для каждого батча.
# batch — кортеж (x, y) из DataLoader.
# x, y = batch — распаковываем изображения и метки.
# logits = self(x) — прямой проход (вызов forward).
# loss = nn.functional.cross_entropy(logits, y) — вычисляем кросс-энтропию.
# preds = torch.argmax(logits, dim=1) — определяем предсказанный класс.
# self.train_acc.update(preds, y) — обновляем счетчик точности.
# self.log("train_loss", loss, prog_bar=True) — логируем значение потерь (будет отображаться в прогресс-баре).
# return loss — возвращаем значение потерь (Lightning использует его для обратного распространения).
    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = nn.functional.cross_entropy(logits, y)
        preds = torch.argmax(logits, dim=1)
        self.train_acc.update(preds, y)
        self.log("train_loss", loss, prog_bar=True)
        return loss

# training_epoch_end вызывается после завершения всей эпохи обучения.
# outs — список результатов (потерь) со всех батчей (в данном коде не используется).
# self.train_acc.compute() — вычисляем итоговую точность за эпоху.
# self.log("train_acc", ...) — логируем точность.
# self.train_acc.reset() — сбрасываем счетчик для следующей эпохи.
    def on_train_epoch_end(self):
        self.log("train_acc", self.train_acc.compute())
        self.train_acc.reset()

# validation_step — аналогичен training_step, но для валидации.
# Логирует valid_loss.
# Обновляет valid_acc.
    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = nn.functional.cross_entropy(logits, y)
        preds = torch.argmax(logits, dim=1)
        self.valid_acc.update(preds, y)
        self.log("valid_loss", loss, prog_bar=True)
        return loss

# После каждой валидационной эпохи вычисляем и логируем точность (valid_acc).
# Сбрасываем счетчик.
    def on_validation_epoch_end(self):
        self.log("valid_acc", self.valid_acc.compute(), prog_bar=True)
        self.valid_acc.reset()
    
# test_step — для финального тестирования модели.
# Отличается тем, что сразу логирует test_acc (а не в test_epoch_end, потому что тестирование обычно один раз).
    def test_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = nn.functional.cross_entropy(logits, y)
        preds = torch.argmax(logits, dim=1)
        self.test_acc.update(preds, y)
        self.log("test_loss", loss, prog_bar=True)
        self.log("test_acc", self.test_acc.compute(), prog_bar=True)
        return loss

# configure_optimizers — Lightning вызывает этот метод, чтобы получить оптимизатор.
# Возвращает Adam со скоростью обучения 0.001.
# Можно также возвращать несколько оптимизаторов или добавлять планировщик скорости (learning rate scheduler), но здесь — только один оптимизатор.
    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=0.001)
        return optimizer

### 13.8.2. Настройка загрузчиков данных для Lightning

Есть три основных способа подготовки набора данных для Lightning. Мы можем:\
♦ сделать набор данных частью модели;\
♦ настроить загрузчики данных как обычно и передать их методу обучения Lightning Trainer - он представлен в следующем разделе;\
♦ создать модуль LightningDataModule.\
Здесь мы воспользуемся модулем LightningDataModule, который предлагает наиболее организованный подход.
LightningDataModule состоит из пяти основных методов, как показано в следующем примере кода: 

In [3]:
from torch.utils.data import DataLoader
from torch.utils.data import random_split
 
from torchvision.datasets import MNIST
from torchvision import transforms

In [4]:
class MnistDataModule(pl.LightningDataModule):
    def __init__(self, data_path='./'):
        super().__init__()
        self.data_path = data_path
        self.transform = transforms.Compose([transforms.ToTensor()])

# prepare_data — метод, который вызывается один раз для скачивания данных. Здесь:
# MNIST(root=self.data_path, download=True) — создается объект датасета, и благодаря download=True данные скачиваются в папку data_path, если их там еще нет.
# Этот метод выполняется только на главном процессе (полезно для распределенного обучения).
# Важно: здесь данные не сохраняются в переменную, они просто скачиваются.  
    def prepare_data(self):
        MNIST(root=self.data_path, download=True) 


# setup — метод, который вызывается на каждом устройстве (например, при распределенном обучении на нескольких GPU).
# stage — указывает, для какой фазы вызывается метод ('fit', 'validate', 'test', 'predict'). Здесь он не используется.
# mnist_all = MNIST(...) — загружаем весь обучающий датасет (60 000 изображений) с трансформацией, но без скачивания (download=False), потому что данные уже скачаны в prepare_data.
# random_split(mnist_all, [55000, 5000], generator=torch.Generator().manual_seed(1)) — разбиваем 60 000 изображений на:
# self.train — 55 000 изображений (обучение)
# self.val — 5 000 изображений (валидация)
# generator=torch.Generator().manual_seed(1) — фиксируем зерно случайности, чтобы разбиение было одинаковым при каждом запуске.
# self.test = MNIST(...) — загружаем тестовый датасет (10 000 изображений) с train=False.
    def setup(self, stage=None):
        mnist_all = MNIST( 
            root=self.data_path,
            train=True,
            transform=self.transform,  
            download=False
        ) 

        self.train, self.val = random_split(
            mnist_all, [55000, 5000], generator=torch.Generator().manual_seed(1)
        )

        self.test = MNIST( 
            root=self.data_path,
            train=False,
            transform=self.transform,  
            download=False
        ) 

# train_dataloader — метод возвращает DataLoader для обучающей выборки.
# batch_size=64 — размер батча.
# num_workers=4 — количество процессов для параллельной загрузки данных.
    def train_dataloader(self):
        return DataLoader(self.train, batch_size=64, num_workers=4)
    
# val_dataloader — возвращает DataLoader для валидационной выборки.
    def val_dataloader(self):
        return DataLoader(self.val, batch_size=64, num_workers=4)
    
# test_dataloader — возвращает DataLoader для тестовой выборки.
    def test_dataloader(self):
        return DataLoader(self.test, batch_size=64, num_workers=4)
    
torch.manual_seed(1) 
mnist_dm = MnistDataModule()

### 13.8.3. Обучение модели с помощью класса PyTorch Lightning Trainer 

Пришло время пожинать плоды своих трудов по настройке модели с помощью именованных методов, а также модуля данных Lightning. Lightning реализует класс тrainer, который делает обучение модели очень удобным, беря на себя все промежуточные шаги - такие как вызовы zero_grad(), reverse() и optimizer.step(). Кроме того, в качестве бонуса он позволяет нам легко указать один или несколько графических процессоров
(если они доступны): 

In [5]:
# Импортируешь класс ModelCheckpoint из подмодуля callbacks PyTorch Lightning.
# Это специальный инструмент (callback), который автоматически сохраняет лучшие версии модели во время обучения.
from pytorch_lightning.callbacks import ModelCheckpoint

mnistclassifier = MultiLayerPerceptron()

# Создаешь список callbacks, в который помещаешь один объект ModelCheckpoint.
# Параметры ModelCheckpoint:
# save_top_k=1 — сохранять только лучшую модель (не несколько, а только одну).
# mode='max' — ищем максимальное значение метрики (чем больше accuracy, тем лучше).
# monitor="valid_acc" — отслеживаем метрику valid_acc (точность на валидации).
# Когда в процессе обучения значение valid_acc становится новым рекордом, модель автоматически сохраняется на диск.
callbacks = [ModelCheckpoint(save_top_k=1, mode='max', monitor="valid_acc")]

# Проверка: доступна ли видеокарта (GPU) через torch.cuda.is_available().
# Если GPU есть: создается объект Trainer с параметрами:
# max_epochs=10 — всего 10 эпох обучения.
# callbacks=callbacks — передаем список колбэков (сохранение лучшей модели).
# gpus=1 — используем одну видеокарту.
# Если GPU нет: создается Trainer без параметра gpus — обучение будет на CPU.
# pl.Trainer — это главный класс Lightning, который управляет всем процессом обучения (циклы, валидация, логирование, устройства и т.д.).
if torch.cuda.is_available(): # if you have GPUs
    trainer = pl.Trainer(max_epochs=10, callbacks=callbacks, accelerator='gpu', devices=1)
else:
    trainer = pl.Trainer(max_epochs=10, callbacks=callbacks)

# Запускаешь обучение!
# model=mnistclassifier — передаем модель.
# datamodule=mnist_dm — передаем DataModule (в котором уже подготовлены датасеты и DataLoader'ы).
# Lightning сам:
# Вызывает prepare_data() (скачивание данных, если нужно).
# Вызывает setup() (разбиение на train/val/test).
# Запускает цикл обучения на 10 эпох.
# На каждой эпохе вызывает training_step, validation_step.
# В конце каждой эпохи вычисляет и логирует метрики.
# Если valid_acc улучшился — сохраняет модель через ModelCheckpoint
trainer.fit(model=mnistclassifier, datamodule=mnist_dm)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
You are using a CUDA device ('NVIDIA GeForce RTX 5060 Laptop GPU') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name      | Type               | Params | Mode  | FLOPs
-----------------------------------------------------------------
0 | train_acc | MulticlassAccuracy | 0      | train | 0    
1 | valid_acc | MulticlassAccuracy | 0      | train | 0    
2 | test_acc  | Multiclass

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

c:\Users\Bushi\OneDrive\Desktop\Scikit_Learn\venv\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
c:\Users\Bushi\OneDrive\Desktop\Scikit_Learn\venv\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:429: Consider setting `persistent_workers=True` in 'val_dataloader' to speed up the dataloader worker initialization.


c:\Users\Bushi\OneDrive\Desktop\Scikit_Learn\venv\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:429: Consider setting `persistent_workers=True` in 'train_dataloader' to speed up the dataloader worker initialization.


Epoch 9: 100%|██████████| 860/860 [00:15<00:00, 56.27it/s, v_num=4, train_loss=0.260, valid_loss=0.166, valid_acc=0.949]  

`Trainer.fit` stopped: `max_epochs=10` reached.


Epoch 9: 100%|██████████| 860/860 [00:15<00:00, 56.26it/s, v_num=4, train_loss=0.260, valid_loss=0.166, valid_acc=0.949]


### 13.8.4. Оценка модели с помощью TensorBoard 

В предыдущем разделе мы убедились в удобстве использования класса тrainer. Еще одна приятная особенность Lightniпg - возможность ведения журнала. Напомним, что ранее мы указали несколько шагов self.log в нашей модели Lightning. После (и даже в процессе) обучения мы можем визуализировать их в TensorBoard. (Заметим, что Lightning также поддерживает другие механизмы журнала. Для получения дополнительной информации обратитесь к официальной документации, доступной по адресу: https://pytorch-lightning.readthedocs.io/enЛatest/commonЛoggers.html.)

По умолчанию Lightning отслеживает обучение в подкаталоге с именем lightning_logs. Чтобы визуализировать обучающие прогоны, выполните следующий код в терминале командной строки, который откроет TensorBoard в вашем браузере:\
tensorboard --logdir lightning_logs/

В качестве альтернативы, если вы запускаете код в блокноте Jupyter, можно добавить
следующий код в ячейку блокнота Jupyter, чтобы отобразить панель инструментов
TensorBoard непосредственно в блокноте:\
%load_ext tensorboard\
%tensorboard --logdir lightning_logs/

На рис. 13.9 показана информационная панель TensorBoard, отображающая точность обучения и проверки. Обратите внимание, что в нижнем левом углу показан переключатель version_0. Если вы запускаете обучающий код несколько раз, Lightning будет сохранять результаты как отдельные вложенные папки: version_0, version_l, version_2 и т. д.

Глядя на графики точности при обучении и валидации, приведенные на рис. 13.9, мы можем предположить, что обучение модели в течение нескольких дополнительных эпох может улучшить ее производительность. 

In [13]:
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [14]:
%tensorboard --logdir lightning_logs/

Reusing TensorBoard on port 6007 (pid 63404), started 1 day, 22:42:05 ago. (Use '!kill 63404' to kill it.)

In [9]:
trainer.test(model=mnistclassifier, datamodule=mnist_dm, ckpt_path='best')

Restoring states from the checkpoint path at c:\Users\Bushi\OneDrive\Desktop\Scikit_Learn\Book\machine-learning-book-main\ch13\lightning_logs\version_4\checkpoints\epoch=8-step=7740.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at c:\Users\Bushi\OneDrive\Desktop\Scikit_Learn\Book\machine-learning-book-main\ch13\lightning_logs\version_4\checkpoints\epoch=8-step=7740.ckpt
c:\Users\Bushi\OneDrive\Desktop\Scikit_Learn\venv\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:429: Consider setting `persistent_workers=True` in 'test_dataloader' to speed up the dataloader worker initialization.


Testing DataLoader 0: 100%|██████████| 157/157 [00:00<00:00, 313.26it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_acc            0.9499600529670715
        test_loss           0.1491229236125946
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


[{'test_loss': 0.1491229236125946, 'test_acc': 0.9499600529670715}]

In [41]:
path = 'lightning_logs/version_0/checkpoints/epoch=8-step=7739.ckpt'

if torch.cuda.is_available(): # if you have GPUs
    trainer = pl.Trainer(
        max_epochs=15, callbacks=callbacks, resume_from_checkpoint=path, gpus=1
    )
else:
    trainer = pl.Trainer(
        max_epochs=15, callbacks=callbacks, resume_from_checkpoint=path
    )

trainer.fit(model=mnistclassifier, datamodule=mnist_dm)

TypeError: Trainer.__init__() got an unexpected keyword argument 'resume_from_checkpoint'

In [42]:
%tensorboard --logdir lightning_logs/

Reusing TensorBoard on port 6007 (pid 63404), started 0:17:22 ago. (Use '!kill 63404' to kill it.)

In [43]:
trainer.test(model=mnistclassifier, datamodule=mnist_dm)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
c:\Users\Bushi\OneDrive\Desktop\Scikit_Learn\venv\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
c:\Users\Bushi\OneDrive\Desktop\Scikit_Learn\venv\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:429: Consider setting `persistent_workers=True` in 'test_dataloader' to speed up the dataloader worker initialization.


Testing DataLoader 0: 100%|██████████| 157/157 [00:00<00:00, 324.79it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_acc            0.9644519686698914
        test_loss           0.12106624990701675
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


[{'test_loss': 0.12106624990701675, 'test_acc': 0.9644519686698914}]

In [44]:
trainer.test(model=mnistclassifier, datamodule=mnist_dm, ckpt_path='best')

ValueError: `.test(ckpt_path="best")` is set but `ModelCheckpoint` is not configured to save the best model.

In [45]:
path = "lightning_logs/version_0/checkpoints/epoch=13-step=12039.ckpt"
model = MultiLayerPerceptron.load_from_checkpoint(path)

FileNotFoundError: [Errno 2] No such file or directory: 'c:/Users/Bushi/OneDrive/Desktop/Scikit_Learn/Book/machine-learning-book-main/ch13/lightning_logs/version_0/checkpoints/epoch=13-step=12039.ckpt'

## Summary

В этой главе мы рассмотрели самые важные и полезные функции PyTorch. Начали мы с обсуждения динамического графа вычислений PyTorch, который делает реализацию вычислений очень удобной. Мы также рассмотрели семантику определения тензорных объектов PyTorch в качестве параметров модели. После знакомства с концепцией вычисления частных производных и градиентов произвольных функций мы более подробно познакомились с модулем torch.nn, предоставляющим нам удобный интерфейс для построения более сложных глубоких нейросетевых моделей. Наконец, мы завершили эту главу, решив задачу регрессии и классификации с помощью новых навыков. Итак, вы изучили основные механизмы PyTorch, а в следующей главе будет представлена концепция архитектуры сверточных нейронных сетей (convolutional neural
network, CNN) для глубокого обучения - мощных моделей, показывающих отличные результаты в области компьютерного зрения. 